# Lekcija 01 - Uvod u AI agente

Dobrodošli na prvu lekciju u tečaju **AI agenti za početnike**!

**AI agent** je program koji koristi veliki jezični model (LLM) kao svoj mehanizam za zaključivanje i može poduzimati *radnje* u stvarnom svijetu — pozivati API-je, upitavati baze podataka ili izvršavati kod — kako bi postigao cilj u ime korisnika.

U ovom bilježniku izgradit ćete svog prvog agenta: **Putničkog agenta** koji preporučuje destinacije za odmor. Usput ćete naučiti kako:

1. Povezati se s Azure AI Foundry Agent servisom koristeći **Microsoft Agent Framework**.
2. Dati agentu **alat** — običnu Python funkciju koju može pozvati.
3. Pokrenuti agenta i pregledati njegov odgovor.
4. Strimirati odgovor agenta token-po-token.


## Postavljanje

Prije pokretanja ovog bilježnika, provjerite imate li:

1. **Azure AI Foundry projekt** s implementiranim modelom za chat (npr. `gpt-4o-mini`).
2. **Prijavljeni ste putem Azure CLI-ja** — pokrenite `az login` u vašem terminalu.
3. **Postavljene potrebne varijable okoline:**
   - `AZURE_AI_PROJECT_ENDPOINT` — vaš Azure AI Foundry projektni endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — naziv vašeg implementiranog modela.

Ćelija ispod instalira Python pakete koje trebate.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Izrada Vašeg Prvog Agenta

Agentu su potrebne dvije stvari:

- **Upute** koje mu govore *tko je* i *kako se ponašati* (sistemski upit).
- **Alati** — Python funkcije označene dekoratorom `@tool` koje agent može pozivati da bi dohvaćao informacije ili izvršavao radnje.

Ispod definiramo jednostavan alat koji vraća popis popularnih destinacija za odmor. Agent će koristiti ovaj alat kada korisnik zatraži preporuke za putovanja.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming odgovori

Za interaktivnije iskustvo možete **streamati** odgovor agenta. Umjesto da čekate puni odgovor, agent isporučuje tekstualne dijelove čim se generiraju. Ovo je posebno korisno u chat sučeljima gdje želite prikazivati izlaz u stvarnom vremenu.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Sažetak

U ovoj lekciji naučili ste kako:

- **Kreirati pružatelja usluge** koji se povezuje na Azure AI Foundry Agent Service putem `AzureAIProjectAgentProvider`.
- **Definirati alat** koristeći dekorator `@tool` kako bi agent mogao pozivati vaše Python funkcije.
- **Pokrenuti agenta** s korisničkom porukom i ispisati njegov odgovor.
- **Prikazivati odgovore u streamu** za ispis u stvarnom vremenu.

U sljedećoj lekciji ćemo detaljnije istražiti agencijske okvire i naučiti kako agentima dati snažnije alate i mogućnosti višestepenog rezoniranja.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
